# AIaaS — Vision-Language Model (VLM) Throughput Benchmark

Portable throughput benchmark for the **multimodal (image + text)** workload — a
VLM answering a prompt about an image. Sweeps batch size and reports decode
throughput, end-to-end latency, and VRAM.

Reports per batch size:
- **output tokens/s** (decode throughput)
- **end-to-end latency** (image+prompt in → full answer out)
- **peak VRAM**

> Tier: *portable proxy* — a clean `transformers` generate() timing run over a
> synthetic image, not an accuracy-gated VQA benchmark. Targets recent
> `transformers` (Qwen2.5-VL via `AutoModelForImageTextToText`).

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate", "pillow", "pandas"], check=False)
import transformers
print("transformers", transformers.__version__)


## 2. Environment & hardware capture

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def gpu_mem_used_mb():
    v = smi("memory.used")
    try:
        return float(v[0]) if v else None
    except Exception:
        return None

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
assert CUDA, "No CUDA GPU detected. This benchmark requires a GPU runtime."
_cc = torch.cuda.get_device_capability(0)

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "compute_capability": f"{_cc[0]}.{_cc[1]}",
    "cuda": torch.version.cuda,
    "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration
VRAM-tiered, ungated Qwen2.5-VL.

In [ ]:
VRAM = ENV["vram_total_gb"]
TIER = "large" if VRAM >= 24 else "small"
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct" if TIER == "large" else "Qwen/Qwen2.5-VL-3B-Instruct"
DTYPE = "bfloat16" if _cc[0] >= 8 else "float16"

BATCH_SIZES = [1, 4]
GEN_TOKENS = 64                 # fixed answer length for comparability
NUM_ITERS = 3                   # timed iterations per batch (after warmup)
IMG_SIZE = 512
PROMPT = "Describe this image in detail."

OUT_DIR = "vlm_bench_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"VRAM={VRAM} GB -> TIER={TIER}, MODEL={MODEL}, dtype={DTYPE}, gen={GEN_TOKENS}")


## 4. Synthetic image + processor inputs

In [ ]:
from PIL import Image
import numpy as np

rng = np.random.default_rng(0)
IMG = Image.fromarray(rng.integers(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

MESSAGES = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": PROMPT},
]}]
print("synthetic image:", IMG.size)


## 5. Load model + run the batch-size sweep

In [ ]:
import time, torch
from transformers import AutoProcessor, AutoModelForImageTextToText

dtype = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16
processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL, torch_dtype=dtype).to("cuda")
model.eval()

prompt_text = processor.apply_chat_template(MESSAGES, tokenize=False, add_generation_prompt=True)

def run_bs(bs):
    torch.cuda.reset_peak_memory_stats()
    inputs = processor(text=[prompt_text] * bs, images=[IMG] * bs, return_tensors="pt").to("cuda")
    def gen():
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=GEN_TOKENS, do_sample=False)
    gen()                                  # warmup
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(NUM_ITERS):
        gen()
    torch.cuda.synchronize()
    dt = time.time() - t0
    out_tokens = bs * GEN_TOKENS * NUM_ITERS
    return {
        "batch_size": bs,
        "output_tokens_per_s": round(out_tokens / dt, 1),
        "latency_s_per_call": round(dt / NUM_ITERS, 3),
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }

RESULTS = []
for bs in BATCH_SIZES:
    try:
        r = run_bs(bs)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); r = {"batch_size": bs, "error": "OOM"}
    except Exception as e:
        r = {"batch_size": bs, "error": f"{type(e).__name__}: {e}"}
    print(r)
    RESULTS.append(r)


## 6. Results — save + summarize

In [ ]:
import pandas as pd
NORM = {
    "schema": "vlm-bench/1.0",
    "env": ENV, "model": MODEL, "dtype": DTYPE,
    "gen_tokens": GEN_TOKENS, "img_size": IMG_SIZE,
    "results": RESULTS,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"]).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"vlm_bench_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("Saved:", out)
print(f"\n==== VLM SUMMARY ====\n{ENV['gpu_name']} | {MODEL} | {DTYPE} | gen={GEN_TOKENS}")
df = pd.DataFrame(RESULTS)
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string(index=False))


## 7. Reading the result
**output tokens/s** at the largest batch that fits is multimodal serving
throughput; **latency_s_per_call** at batch 1 is best-case response time. Note
VLMs spend a chunk of the prefill on image tokens, so latency is higher than a
text-only LLM at the same answer length. Pairs with the LLM serving notebook for
a multimodal-vs-text capacity comparison.